# ISIC 2024 - Trích xuất đặc trưng Khái niệm lâm sàng bằng MONET
Notebook này sử dụng mô hình nền móng đa phương thức **MONET (Medical cONcept rETriever)** để trích xuất các đặc trưng khái niệm y khoa từ ảnh tổn thương da của cuộc thi ISIC 2024.

### Tính năng tự động nhận diện môi trường:
- **Khi chạy trên Kaggle:** Tự động tìm kiếm ảnh trong `/kaggle/input/competitions/isic-2024-challenge` hoặc `/kaggle/input/isic-2024-challenge` và xuất file CSV ra `/kaggle/working/`.
- **Khi chạy ở Local (máy cá nhân):** Tự động chuyển đường dẫn về thư mục `D:\HK1_2025_2026\SEMINAR_CODE\data` hoặc các thư mục tương đương.

### Hướng dẫn chạy trên Kaggle:
1. Bật **GPU T4 x2** hoặc **GPU P100** trong cài đặt Notebook (Settings -> Accelerator -> GPU).
2. Bật cài đặt **Internet** ở bảng bên phải để tải trọng số mô hình MONET từ HuggingFace.
3. Chạy toàn bộ các ô (Run All). Đầu ra sẽ là 2 tệp CSV: `train_monet_features.csv` và `test_monet_features.csv` trong thư mục làm việc.

In [ ]:
import io
import h5py
import torch
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel

# Kiểm tra thiết bị GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Thiết bị đang sử dụng: {device}")

In [ ]:
# ==========================================
# CELL 2: CẤU HÌNH TỰ ĐỘNG DÒ TÌM ĐƯỜNG DẪN
# ==========================================

RUN_MODE = 'auto'  # 'auto' | 'kaggle' | 'local'

# ===== KAGGLE PATHS =====
KAGGLE_COMP_DATASET_ROOT = '/kaggle/input/competitions/isic-2024-challenge'
KAGGLE_COMP_DATASET_ROOT_ALT = '/kaggle/input/isic-2024-challenge'
KAGGLE_OUTPUT_DIR = '/kaggle/working'

# ===== LOCAL PATHS =====
LOCAL_DATA_ROOT = r"D:\HK1_2025_2026\SEMINAR_CODE\data"
LOCAL_OUTPUT_DIR = '.'

def _first_existing(candidates):
    for candidate in candidates:
        p = Path(candidate)
        if p.exists():
            return p
    return None

if RUN_MODE == 'auto':
    is_kaggle = Path('/kaggle/input').exists()
else:
    is_kaggle = RUN_MODE == 'kaggle'

if is_kaggle:
    # Dò tìm thư mục chứa dữ liệu cuộc thi trên Kaggle
    comp_root = _first_existing([KAGGLE_COMP_DATASET_ROOT, KAGGLE_COMP_DATASET_ROOT_ALT])
    if comp_root is None:
        comp_root = Path(KAGGLE_COMP_DATASET_ROOT_ALT)
        
    TRAIN_HDF5_PATH = str(comp_root / 'train-image.hdf5')
    TEST_HDF5_PATH = str(comp_root / 'test-image.hdf5')
    TRAIN_OUTPUT_PATH = str(Path(KAGGLE_OUTPUT_DIR) / 'train_monet_features.csv')
    TEST_OUTPUT_PATH = str(Path(KAGGLE_OUTPUT_DIR) / 'test_monet_features.csv')
else:
    # Dò tìm thư mục chứa dữ liệu ở máy local
    if LOCAL_DATA_ROOT is not None and Path(LOCAL_DATA_ROOT).exists():
        local_root = Path(LOCAL_DATA_ROOT)
    else:
        local_root = _first_existing(['./data', '../data', '.'])
        if local_root is None:
            local_root = Path('.')
            
    TRAIN_HDF5_PATH = str(local_root / 'train-image.hdf5')
    TEST_HDF5_PATH = str(local_root / 'test-image.hdf5')
    TRAIN_OUTPUT_PATH = str(Path(LOCAL_OUTPUT_DIR) / 'train_monet_features.csv')
    TEST_OUTPUT_PATH = str(Path(LOCAL_OUTPUT_DIR) / 'test_monet_features.csv')

print('✅ Môi trường nhận diện:', 'KAGGLE' if is_kaggle else 'LOCAL')
print('✅ Đường dẫn ảnh Train HDF5:', TRAIN_HDF5_PATH)
print('✅ Đường dẫn ảnh Test HDF5:', TEST_HDF5_PATH)
print('✅ Đường dẫn xuất file Train CSV:', TRAIN_OUTPUT_PATH)
print('✅ Đường dẫn xuất file Test CSV:', TEST_OUTPUT_PATH)

In [ ]:
# Tải mô hình MONET
print("Đang tải mô hình MONET từ HuggingFace...")
model = CLIPModel.from_pretrained("suinleelab/MONET").to(device)
processor = CLIPProcessor.from_pretrained("suinleelab/MONET")
model.eval()
print("Đã tải mô hình thành công!")

In [ ]:
# ==========================================================
# HÀM PHỤ TRỢ: TRÍCH XUẤT TENSOR AN TOÀN TỪ ĐẦU RA MÔ HÌNH
# ==========================================================
def get_tensor_features(outputs, model, mode="text"):
    """
    Trích xuất Tensor thô từ output của HuggingFace, tự động xử lý 
    nếu kết quả trả về là BaseModelOutputWithPooling thay vì Tensor.
    Đồng thời áp dụng layer chiếu tuyến tính (Projection) của CLIP để căn chỉnh không gian nhúng.
    Có kiểm tra kích thước động để tránh lỗi lệch chiều ma trận giữa các layer.
    """
    if isinstance(outputs, torch.Tensor):
        return outputs
        
    # Trường hợp 1: Có sẵn thuộc tính nhúng cụ thể
    if mode == "text" and hasattr(outputs, "text_embeds"):
        return outputs.text_embeds
    if mode == "image" and hasattr(outputs, "image_embeds"):
        return outputs.image_embeds
        
    # Trường hợp 2: Trả về đối tượng chứa pooler_output (như BaseModelOutputWithPooling)
    if hasattr(outputs, "pooler_output"):
        pooled = outputs.pooler_output
        proj_attr = "text_projection" if mode == "text" else "visual_projection"
        if hasattr(model, proj_attr):
            proj_layer = getattr(model, proj_attr)
            # Chỉ thực hiện chiếu tuyến tính nếu số chiều đầu vào (in_features) trùng khớp với kích thước của pooled
            if proj_layer is not None and hasattr(proj_layer, "in_features"):
                if proj_layer.in_features == pooled.shape[-1]:
                    return proj_layer(pooled)
        return pooled
        
    # Trường hợp 3: Trả về Tuple/List có chứa pooler_output ở vị trí số 1
    try:
        if len(outputs) > 1:
            pooled = outputs[1]
            proj_attr = "text_projection" if mode == "text" else "visual_projection"
            if hasattr(model, proj_attr):
                proj_layer = getattr(model, proj_attr)
                if proj_layer is not None and hasattr(proj_layer, "in_features"):
                    if proj_layer.in_features == pooled.shape[-1]:
                        return proj_layer(pooled)
            return pooled
    except:
        pass
        
    # Trường hợp 4: Fallback về phần tử đầu tiên hoặc giữ nguyên
    try:
        return outputs[0]
    except:
        return outputs

In [ ]:
# Định nghĩa danh sách các khái niệm lâm sàng
concepts = [
    "atypical pigment network",
    "blue white veil",
    "ulceration",
    "irregular streaks",
    "milky red area",
    "irregular dots",
    "asymmetric border",
    "scales or hyperkeratosis",
    "regression structure",
    "homogeneous pigmentation"
]

text_prompts = [f"an image of a skin lesion with {concept}" for concept in concepts]
print("Các Prompts sử dụng để đối chiếu:")
for p in text_prompts:
    print(f" - {p}")

In [ ]:
# Tính toán trước đặc trưng văn bản (Text Embeddings) để tiết kiệm thời gian
inputs_text = processor(text=text_prompts, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    outputs = model.get_text_features(**inputs_text)
    # Sử dụng hàm get_tensor_features để trích xuất và chiếu không gian vector chuẩn xác
    text_features = get_tensor_features(outputs, model, mode="text")
    text_features = text_features / text_features.norm(dim=-1, keepdim=True) # Chuẩn hóa L2
print("Đã mã hóa và chuẩn hóa vector văn bản của khái niệm!")

In [ ]:
# Định nghĩa Dataset PyTorch để tối ưu tốc độ đọc ảnh song song bằng DataLoader
class ISICHDF5Dataset(Dataset):
    def __init__(self, hdf5_path, processor):
        self.hdf5_path = hdf5_path
        self.processor = processor
        # Đọc danh sách khóa (ids) từ HDF5
        with h5py.File(self.hdf5_path, "r") as f:
            self.isic_ids = list(f.keys())
        self.h5_file = None

    def __len__(self):
        return len(self.isic_ids)

    def __getitem__(self, idx):
        # Tránh lỗi đa luồng trong PyTorch DataLoader bằng cách chỉ khởi tạo HDF5 khi luồng bắt đầu
        if self.h5_file is None:
            self.h5_file = h5py.File(self.hdf5_path, "r")
            
        isic_id = self.isic_ids[idx]
        raw_data = self.h5_file[isic_id][()]
        
        # Giải nén ảnh và đưa về định dạng RGB
        image = Image.open(io.BytesIO(raw_data)).convert("RGB")
        
        # Tiền xử lý ảnh thông qua CLIP Processor
        pixel_values = self.processor(images=image, return_tensors="pt")["pixel_values"].squeeze(0)
        
        return isic_id, pixel_values

In [ ]:
# Hàm trích xuất đặc trưng theo lô (Batch Processing)
def extract_features_to_csv(hdf5_path, csv_output_path, batch_size=128):
    print(f"Đang khởi tạo Dataset từ: {hdf5_path}")
    dataset = ISICHDF5Dataset(hdf5_path, processor)
    
    # Sử dụng num_workers=2 để tận dụng tối đa CPU của Kaggle giải nén ảnh song song
    dataloader = DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=2, 
        pin_memory=True
    )
    
    results = []
    
    with torch.no_grad():
        for ids, pixels in tqdm(dataloader, desc="Đang trích xuất đặc trưng"):
            pixels = pixels.to(device)
            
            # Trích xuất đặc trưng ảnh
            img_outputs = model.get_image_features(pixel_values=pixels)
            # Trích xuất và chiếu không gian vector ảnh
            img_features = get_tensor_features(img_outputs, model, mode="image")
            img_features = img_features / img_features.norm(dim=-1, keepdim=True)
            
            # Tính toán độ tương đồng Cosine
            similarities = (img_features @ text_features.T).cpu().numpy()
            
            for isic_id, sim in zip(ids, similarities):
                results.append([isic_id] + list(sim))
                
    # Tạo DataFrame và lưu
    columns = ["isic_id"] + [f"monet_{c.replace(' ', '_')}" for c in concepts]
    df_out = pd.DataFrame(results, columns=columns)
    df_out.to_csv(csv_output_path, index=False)
    print(f"Đã lưu kết quả thành công vào: {csv_output_path}\n")

In [ ]:
# 1. Xử lý tập huấn luyện (Train Data)
extract_features_to_csv(TRAIN_HDF5_PATH, TRAIN_OUTPUT_PATH, batch_size=128)

# 2. Xử lý tập kiểm thử (Test Data)
extract_features_to_csv(TEST_HDF5_PATH, TEST_OUTPUT_PATH, batch_size=128)

In [ ]:
# Kiểm tra nhanh kết quả vừa sinh ra
df_check = pd.read_csv(TRAIN_OUTPUT_PATH)
print("Hình dáng tập dữ liệu đặc trưng trích xuất:", df_check.shape)
print("5 dòng đầu tiên của đặc trưng MONET:")
df_check.head()